In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import os, glob, json
from pathlib import Path
import numpy as np
import jax
from jax import value_and_grad, jit, vmap, grad, random
import jax.numpy as jnp
import flax.linen as nn
import jax.lax as lax
from flax import serialization
import matplotlib.pyplot as plt
import optax

from action_angle_networks.sk_models import MyActionAngleNetwork
from functools import partial
from dataclasses import dataclass
from typing import Any, Callable, Sequence, Optional, List, Tuple, Dict

%config InlineBackend.figure_format = 'retina'


In [3]:
import sys
from pathlib import Path

Path.cwd()

# SRC = (Path.cwd() / "kamiya_asymmetry" / "src").resolve() # /workspace/kamiya_asymmetry/src
# sys.path.insert(0, str(SRC))    


PosixPath('/work/gb20/b20109')

# Toda lattice

We consider the (open or periodic) Toda lattice
\begin{equation}
H(q, p)= \sum_{i=1}^n \frac{p_i^2}{2m} + a \sum_{i} \exp igl(-(q_{i+1}-q_i) igr),
\end{equation}
with the equations of motion
\begin{align}
\dot q_i &= \frac{p_i}{m},\\
\dot p_i &= a\,\Bigl( e^{-(q_i - q_{i-1})} - e^{-(q_{i+1} - q_i)} \Bigr).
\end{align}

We use a leapfrog (symplectic) integrator to generate trajectories as the training data.


# Define classes
To sample multiple initial values, we first rewrite the code into variable initial values.


In [4]:
n = 4  # number of lattice sites
T = 1500  # number of integration steps for inspection plots
dt = 0.01  # time step


In [5]:
seed = 42
rng = np.random.default_rng(seed)
alpha = 1.0


In [6]:
@dataclass
class TodaParams:
    n: int
    alpha: float = 1.0
    periodic: bool = True
    mass: float = 1.0


def set_toda_params(n: int, alpha: float = 1.0, periodic: bool = True, mass: float = 1.0) -> TodaParams:
    return TodaParams(n=int(n), alpha=float(alpha), periodic=bool(periodic), mass=float(mass))


def toda_hamiltonian(q: jnp.ndarray, p: jnp.ndarray, params: TodaParams) -> jnp.ndarray:
    """Hamiltonian for batch inputs.
    q, p: (B, n) -> returns (B,)
    """
    if isinstance(params, dict):
        alpha = float(params.get("alpha", 1.0))
        mass = float(params.get("mass", 1.0))
        periodic = bool(params.get("periodic", False))
        n_loc = int(params.get("n", q.shape[-1]))
    else:
        alpha = float(params.alpha)
        mass = float(params.mass)
        periodic = bool(params.periodic)
        n_loc = int(params.n)
    q = jnp.atleast_2d(jnp.asarray(q))
    p = jnp.atleast_2d(jnp.asarray(p))
    kin = 0.5 * jnp.sum((p ** 2) / mass, axis=1)
    if n_loc > 1:
        diff = q[:, 1:] - q[:, :-1]
        pot = alpha * jnp.sum(jnp.exp(-diff), axis=1)
    else:
        pot = jnp.zeros(q.shape[0])
    if periodic and n_loc > 1:
        pot = pot + alpha * jnp.exp(-(q[:, 0] - q[:, -1]))
    return kin + pot


@partial(jax.jit, static_argnames=("params",))
def toda_force(q: jnp.ndarray, params: TodaParams) -> jnp.ndarray:
    """Returns dp/dt for q (and shared params).
    q: (B, n)
    """
    alpha = float(params.alpha)
    periodic = bool(params.periodic)
    q = jnp.atleast_2d(q)
    B, n_loc = q.shape
    dp = jnp.zeros_like(q)
    if n_loc > 1:
        exp_forward = alpha * jnp.exp(-(q[:, 1:] - q[:, :-1]))  # (B, n-1)
        dp = dp.at[:, :-1].add(-exp_forward)
        dp = dp.at[:, 1:].add(exp_forward)
        if periodic:
            wrap = alpha * jnp.exp(-(q[:, 0] - q[:, -1]))
            dp = dp.at[:, 0].add(wrap)
            dp = dp.at[:, -1].add(-wrap)
    return dp


@partial(jax.jit, static_argnames=("params",))
def leapfrog_step(q: jnp.ndarray, p: jnp.ndarray, dt: float, params: TodaParams) -> Tuple[jnp.ndarray, jnp.ndarray]:
    mass = float(params.mass)
    p_half = p + 0.5 * dt * toda_force(q, params)
    q_new = q + dt * p_half / mass
    p_new = p_half + 0.5 * dt * toda_force(q_new, params)
    return q_new, p_new


@partial(jax.jit, static_argnames=("params",))
def simulate_toda(
    params: TodaParams,
    num_steps: int,
    dt: float,
    q0: jnp.ndarray,
    p0: jnp.ndarray,
) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, dict]:
    """Integrate the Toda lattice with leapfrog for `num_steps`.
    Returns trajectories (T+1, B, n).
    """
    num_steps = int(num_steps)
    q0 = jnp.atleast_2d(jnp.asarray(q0, dtype=jnp.float64))
    p0 = jnp.atleast_2d(jnp.asarray(p0, dtype=jnp.float64))

    def body(carry, _):
        q, p = carry
        q, p = leapfrog_step(q, p, dt, params)
        return (q, p), (q, p)

    (_, _), (q_hist, p_hist) = lax.scan(body, (q0, p0), jnp.arange(num_steps))
    q_traj = jnp.concatenate([q0[None, ...], q_hist], axis=0)
    p_traj = jnp.concatenate([p0[None, ...], p_hist], axis=0)
    t = jnp.arange(num_steps + 1, dtype=jnp.float64) * float(dt)

    B = q_traj.shape[1]
    energies = toda_hamiltonian(q_traj.reshape((-1, params.n)), p_traj.reshape((-1, params.n)), params)
    aux = {"energy": energies.reshape(num_steps + 1, B)}
    return t, q_traj, p_traj, aux


In [7]:
params = set_toda_params(n=n, alpha=alpha, periodic=True, mass=1.0)

q0 = rng.normal(scale=0.2, size=(1, n))
p0 = rng.normal(scale=0.2, size=(1, n))

t, q_traj, p_traj, aux = simulate_toda(
    params=params, num_steps=T, dt=dt, q0=q0, p0=p0
)

energy = np.asarray(aux["energy"]).squeeze()
q_np = np.asarray(q_traj).squeeze()
p_np = np.asarray(p_traj).squeeze()

q_np.shape, p_np.shape, energy.shape


ValueError: Non-hashable static arguments are not supported. An error occurred while trying to hash an object of type <class '__main__.TodaParams'>, TodaParams(n=4, alpha=1.0, periodic=True, mass=1.0). The error was:
TypeError: unhashable type: 'TodaParams'


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(t, q_np)
axes[0].set_ylabel('q')
axes[0].set_title('Toda lattice trajectories (leapfrog)')
axes[0].grid(True)

axes[1].plot(t, p_np)
axes[1].set_ylabel('p')
axes[1].grid(True)

axes[2].plot(t, energy, label='H(q,p)')
axes[2].axhline(energy[0], color='k', linestyle='--', linewidth=1, label='initial')
axes[2].set_xlabel('time')
axes[2].set_ylabel('Hamiltonian')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()


# Prepare the dataset
Here we settle up the dataset for training. Let $x=(q, p) \in X,\; \dot{x} = (\dot{q}, \dot{p})$, and the flow $\varphi_t: X 	o X$ follow $\dot{x} = J\nabla_xH$. We define the dataset $\mathcal{D}$ as
\begin{equation}
\mathcal{D} = \{(x_0, \dot{x}_0, x(t), t) \mid x_0 \in X,\; \dot{x}_0 = J\nabla H(x_0),\; x(t) = \varphi_t(x),\; t\in\mathbb{R}_{>0}\}.
\end{equation}

To create such a dataset (technically, a finite subset of the dataset define above), we sample 
+ $x_0$ from a compact set
+ $t$ with an upper bount $t_{\text{max}}$


In [8]:
hyper_params = set_toda_params(n=n, alpha=alpha, periodic=True, mass=1.0)


In [9]:
def create_dataset(x0, steps, dt=dt, hyper_params=hyper_params):
    """
    x0: (B, 2n) initial conditions (q0 | p0)
    steps: (B,) integer time indices at which to sample the flow
    Returns:
      x0: (B, 2n)
      x0_dot: (B, 2n)
      xt: (B, 2n)
      t: (B,)
    """
    x0 = jnp.asarray(x0)
    steps = jnp.asarray(steps).astype(int)

    B, dim = x0.shape
    n_loc = dim // 2
    q0 = x0[:, :n_loc]
    p0 = x0[:, n_loc:]

    max_step = int(steps.max(initial=0))
    t_full, q_full, p_full, _ = simulate_toda(
        params=hyper_params,
        num_steps=max_step,
        dt=dt,
        q0=q0,
        p0=p0,
    )

    batch_idx = jnp.arange(B)
    q_sel = q_full[steps, batch_idx, :]
    p_sel = p_full[steps, batch_idx, :]
    xt = jnp.concatenate([q_sel, p_sel], axis=1)  # (B, 2n)
    t_sel = t_full[steps]

    dqdt0 = p0 / hyper_params.mass  # (B, n)
    dpdt0 = toda_force(q0, hyper_params)
    x0_dot = jnp.concatenate([dqdt0, dpdt0], axis=1)  # (B, 2n)

    return x0, x0_dot, xt, t_sel


N_train = 5000
N_test  = 5000
delta_t_max = 0.1
train_t_dist = "LNP" # "uniform", "step_scheduling", "LNP", "step_scheduling_over_all_epochs"

# delta_t_max_test = 0.5

# Below ignored unless "LNP"
delta_t_mean = 0.1
sigma2 = 0.2

key = jax.random.PRNGKey(0)
q_train_key, p_train_key, dt_train_key, key = jax.random.split(key, 4)
q0_train = jax.random.uniform(q_train_key, shape=(N_train, n), minval=-1.0, maxval=1.0)
p0_train = jax.random.uniform(p_train_key, shape=(N_train, n), minval=-1.0, maxval=1.0)

if train_t_dist == "uniform":
    delta_t_train = jnp.clip(
        dt * jax.random.randint(dt_train_key, shape=(N_train,), minval=1, maxval=delta_t_max),
        min=dt,
        max=delta_t_max
    )
elif train_t_dist == "step_scheduling":
    unif_vec = jax.random.uniform(dt_train_key, shape=(N_train,), minval=0.0, maxval=1.0)
    delta_t_train = jnp.clip(
        dt * jnp.ceil( unif_vec * ((delta_t_max * jnp.arange(N_train)/N_train)//dt)),
        min=dt,
        max=delta_t_max
    )
elif train_t_dist == "LNP":
    sigma2 =  1.0 / 2.0
    dt_train_key, dt_train_key2 = jax.random.split(dt_train_key, 2)
    tau = jnp.sqrt(sigma2) * jax.random.normal( key=dt_train_key, shape=(N_train,) ) + jnp.log( float(delta_t_mean//dt-1) ) - 0.5 * sigma2
    # tau ~ N(log(T_mean-1)-0.5*sigma^2, sigma)
    delta_t_train = dt* (jax.random.poisson(key=dt_train_key2, lam=jnp.exp(tau)) + 1)

step_counts_train = jnp.maximum(1, jnp.round(delta_t_train / dt).astype(int))
train_dataset = create_dataset(
    x0=jnp.concatenate([q0_train, p0_train], axis=1),
    steps=step_counts_train,
    dt=dt,
    hyper_params=hyper_params
)

q_test_key, p_test_key, dt_test_key = jax.random.split(key, 3)
q0_test = jax.random.uniform(q_test_key, shape=(N_test, n), minval=-1.0, maxval=1.0)
p0_test = jax.random.uniform(p_test_key, shape=(N_test, n), minval=-1.0, maxval=1.0)

steps_to_test = [1,2,5,10,20,50]

delta_t_test = jnp.tile(jnp.array(steps_to_test, dtype=int), reps=N_test) * dt
step_counts_test = jnp.tile(jnp.array(steps_to_test, dtype=int), reps=N_test)

test_dataset = create_dataset(
    x0=jnp.concatenate([jnp.repeat(q0_test, repeats=len(steps_to_test), axis=0), jnp.repeat(p0_test, repeats=len(steps_to_test), axis=0)], axis=1),
    steps=step_counts_test,
    dt=dt,
    hyper_params=hyper_params
)


ValueError: Non-hashable static arguments are not supported. An error occurred while trying to hash an object of type <class '__main__.TodaParams'>, TodaParams(n=4, alpha=1.0, periodic=True, mass=1.0). The error was:
TypeError: unhashable type: 'TodaParams'


In [ ]:
plt.plot(delta_t_train)


In [ ]:
test_dataset[0].shape


# Build model & train


In [ ]:
model = MyActionAngleNetwork(
    dim_config=n,
    dim_hidden=128,
    dim_hidden_list=[32,128, 128, 128, 32],
    num_gsblocks=30,
    type_polar="canonical",
    activation=nn.silu,
    mlp_res_connection=False,
    theta_predictor="gradient",
    learn_scale=False
)


In [ ]:
rng = jax.random.PRNGKey(0)
q0 = jnp.zeros((1, int(n)))
p0 = jnp.zeros((1, int(n)))
delta_t0 = jnp.asarray(0.1)

variables = model.init(rng, q0, p0, delta_t0)
params = variables['params']
num_params = sum(jax.tree_util.tree_leaves(jax.tree_util.tree_map(lambda x: x.size, params)))
print(f'Total parameters: {num_params}')


# Training


In [ ]:
def train_step(params,
               opt_state,
               dtst_batch,
               apply_fn,
               tx,
               loss_weights=(1.0, 1.0, 0.0, 0.0),
               action_loss_type='var'):
    x, x_dot, y, delta_t = dtst_batch  # x, x_dot, y: (B, 2n), delta_t: (B,)    
    
    n = x.shape[1] // 2
    q, p = x[:, :n], x[:, n:]  # (B, n), (B, n)
    q_dot, p_dot = x_dot[:, :n], x_dot[:, n:]

    
    def loss_fn(pp):
        q_, p_, I, _ = apply_fn({'params': pp}, q, p, delta_t, train=True)

        loss_q_se = ((1./(1.+delta_t))*(((q_ - y[:, :n]) ** 2).sum(axis=1))).sum()
        loss_p_se = ((1./(1.+delta_t))*(((p_ - y[:, n:]) ** 2).sum(axis=1))).sum()
        
        def H_hat_single(qi, pi):
            return apply_fn({'params': pp}, qi[None, :], pi[None, :],
                            method=model.hamiltonian).squeeze()
        
        def H_of_I_sum(I_in):
            return apply_fn({'params': pp}, I_in, method=model.h_of_I).sum()

        omega = jax.grad(H_of_I_sum)(I)  # (B,n)
        omega_norm = jnp.mean(jnp.linalg.norm(omega, axis=1))  # スカラー

        dHdq_model = jax.vmap(lambda qi, pi: jax.grad(H_hat_single, argnums=0)(qi, pi))(q, p)
        dHdp_model = jax.vmap(lambda qi, pi: jax.grad(H_hat_single, argnums=1)(qi, pi))(q, p)

        L_HNN = ((dHdq_model + p_dot) ** 2).mean() + ((dHdp_model - q_dot) ** 2).mean()

        if action_loss_type=='var':
            L_action = lax.cond(
                loss_weights[2] > 0.0,
                lambda I: jnp.var(I, axis=0).sum(),
                lambda I: 0.0,
                I
            )
        elif action_loss_type=='quadratic_variation':
            L_action = jnp.sum(jnp.square(I[1:, :] - I[:-1, :])) / I.shape[0]

        return (loss_weights[0] * loss_q_se +
                loss_weights[1] * loss_p_se +
                loss_weights[2] * L_action +
                loss_weights[3] * L_HNN), omega_norm

    (loss, omega_norm), grads = value_and_grad(loss_fn, has_aux=True)(params)
    updates, opt_state = tx.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    grad_norm = optax.global_norm(grads)
    return params, opt_state, loss, grad_norm, omega_norm


train_step_jit = jax.jit(train_step, static_argnames=('apply_fn', 'tx'))


@partial(jax.jit, static_argnames=('apply_fn',))
def eval_batch(params, x, y, delta_t, *, apply_fn):
    n = x.shape[1] // 2
    q, p = x[:, :n], x[:, n:]

    q_, p_, _, _ = apply_fn({'params': params}, q, p, delta_t, train=False)

    loss_q_se = ((1./(1.+delta_t))*(((q_ - y[:, :n]) ** 2).sum(axis=1))).sum()
    loss_p_se = ((1./(1.+delta_t))*(((p_ - y[:, n:]) ** 2).sum(axis=1))).sum()
    
    return loss_q_se + loss_p_se


In [ ]:
seed = 201
lr = 1e-3
batch_size = 256


epochs = 200
ep_shuffle_chunks = 4

log_every = 50
loss_weights = (1.0, 1.0, 0.0, 0.0)


key = jax.random.PRNGKey(seed)
variables = model.init(key, jnp.zeros((1, n)), jnp.zeros((1, n)), jnp.asarray(dt))
nn_params = variables['params']

tx = optax.chain(
    optax.clip_by_global_norm(max_norm=1.0),  # Clip gradients by their global norm
    optax.adam(learning_rate=lr)            # Use Adam optimizer
)
opt_state = tx.init(nn_params)

idx_serial_batch = 0
batch_losses = []
grad_norms = []
omega_norms = []

for ep in range(epochs):
    
    if train_t_dist == "step_scheduling":
        if ep == 0:
            x0_train, x0_dot_train, xt_train, t_train = train_dataset
        else:
            for idx_ep in range(ep_shuffle_chunks):
                perm_key, key = jax.random.split(key)
                perm = jax.random.permutation(perm_key, N_train // ep_shuffle_chunks)
                start_idx = idx_ep * (N_train // ep_shuffle_chunks)
                end_idx = (idx_ep + 1) * (N_train // ep_shuffle_chunks)
                x0_train, x0_dot_train, xt_train, t_train = train_dataset
                x0_train = x0_train.at[start_idx:end_idx].set(x0_train[start_idx:end_idx][perm])
                x0_dot_train = x0_dot_train.at[start_idx:end_idx].set(x0_dot_train[start_idx:end_idx][perm])
                xt_train = xt_train.at[start_idx:end_idx].set(xt_train[start_idx:end_idx][perm])
                t_train = t_train.at[start_idx:end_idx].set(t_train[start_idx:end_idx][perm])
    else:
        perm_key, key = jax.random.split(key)
        perm = jax.random.permutation(perm_key, N_train)
        x0_train, x0_dot_train, xt_train, t_train = train_dataset
        x0_train = x0_train[perm]
        x0_dot_train = x0_dot_train[perm]
        xt_train = xt_train[perm]
        t_train = t_train[perm]
    
    num_batches = N_train // batch_size
    epoch_loss = 0.0

    for i in range(num_batches):
        batch_x0     = x0_train[i*batch_size:(i+1)*batch_size]
        batch_x0_dot = x0_dot_train[i*batch_size:(i+1)*batch_size]
        batch_xt     = xt_train[i*batch_size:(i+1)*batch_size]
        batch_t      = t_train[i*batch_size:(i+1)*batch_size]

        nn_params, opt_state, batch_loss, grad_norm, omega_norm = train_step_jit(
            nn_params, opt_state,
            (batch_x0, batch_x0_dot, batch_xt, batch_t),
            apply_fn=model.apply, tx=tx,
            loss_weights=loss_weights
        )
        total_batch_loss = batch_loss * len(batch_t)
        epoch_loss += total_batch_loss
        batch_losses.append(batch_loss)
        grad_norms.append(grad_norm)
        omega_norms.append(omega_norm)

        if (idx_serial_batch + 1) % log_every == 0:
            print(f"Epoch {ep+1}, Batch {i+1}/{num_batches}, Loss: {batch_loss:.6f}")

        if (idx_serial_batch + 1) % log_every == 0 or idx_serial_batch == 1 or (ep==epochs-1 and i==num_batches-1):
            test_losses = []
            num_test_batches = N_test // batch_size
            for j in range(num_test_batches):
                batch_x0_test = test_dataset[0][j*batch_size:(j+1)*batch_size]
                batch_x0_dot_test = test_dataset[1][j*batch_size:(j+1)*batch_size]
                batch_xt_test = test_dataset[2][j*batch_size:(j+1)*batch_size]
                batch_t_test = test_dataset[3][j*batch_size:(j+1)*batch_size]

                test_loss = eval_batch(
                    nn_params,
                    batch_x0_test,
                    batch_xt_test,
                    batch_t_test,
                    apply_fn=model.apply
                )
                test_losses.append(test_loss * len(batch_t_test))
            total_test_loss = sum(test_losses) / N_test
            print(f"*** Test Loss after {idx_serial_batch} batches: {total_test_loss:.6f} ***")
        
        idx_serial_batch += 1

    epoch_loss /= N_train
    print(f"Epoch {ep+1} completed. Average Loss: {epoch_loss:.6f}")


In [ ]:
ax, fig = plt.subplots(figsize=(8, 5))
fig.plot(batch_losses)
fig.set_ylim(0.0, 1.0)
fig.set_xlabel('Batch number')
fig.set_ylabel('Training Loss')
fig.set_title('Training Loss over Batches')
fig.grid()
plt.show()


In [ ]:
ax, fig = plt.subplots(figsize=(8, 5))
fig.plot(grad_norms)
fig.set_yscale('log')
fig.set_xlabel('Batch number')
fig.set_ylabel('Gradient Norm')
fig.set_title('Gradient Norm over Batches')
fig.grid()
plt.show()


In [ ]:
ax, fig = plt.subplots(figsize=(8, 5))
fig.plot(omega_norms)
fig.set_yscale('log')
fig.set_xlabel('Batch number')
fig.set_ylabel('Omega = ∂H/∂I norm')
fig.set_title('Omega Norm over Batches')
fig.grid()
plt.show()


In [ ]:
test_dataset[0][0:1, :n]


In [ ]:
T_total = 200  # number of rollout steps for qualitative check

q0 = test_dataset[0][0:1, :n]  # shape (1, n)
p0 = test_dataset[0][0:1, n:]

t_vec, q_true, p_true, _ = simulate_toda(
    params=hyper_params,
    num_steps=T_total,
    dt=dt,
    q0=q0,
    p0=p0,
)
q_true = q_true.squeeze()
p_true = p_true.squeeze()


def step_fn(carry, _):
    q_curr, p_curr = carry
    q_next, p_next, _, _ = model.apply(
        {'params': nn_params}, q_curr, p_curr, dt, train=False
    )
    return (q_next, p_next), (q_next, p_next)

if T_total <= 0:
    q_pred = q0
    p_pred = p0
else:
    (_, _), (qs_seq, ps_seq) = jax.lax.scan(
        step_fn, (q0, p0), None, length=T_total
    )
    if qs_seq.ndim == 3:
        qs_seq = jnp.squeeze(qs_seq, axis=1)
        ps_seq = jnp.squeeze(ps_seq, axis=1)
    q_pred = jnp.concatenate([q0, qs_seq], axis=0)
    p_pred = jnp.concatenate([p0, ps_seq], axis=0)

t_arr = np.asarray(t_vec)[:T_total+1]
q_true_np = np.asarray(q_true)[:T_total+1]
p_true_np = np.asarray(p_true)[:T_total+1]
q_pred_np = np.asarray(q_pred)
p_pred_np = np.asarray(p_pred)

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

for i in range(q_pred_np.shape[1]):
    axes[0].plot(t_arr, q_true_np[:, i], label=f'true q[{i}]', alpha=0.7)
    axes[0].plot(t_arr, q_pred_np[:, i], '--', label=f'pred q[{i}]', alpha=0.9)
axes[0].set_ylabel('q')
axes[0].grid(True)
axes[0].legend(ncol=2, fontsize=8)

for i in range(p_pred_np.shape[1]):
    axes[1].plot(t_arr, p_true_np[:, i], label=f'true p[{i}]', alpha=0.7)
    axes[1].plot(t_arr, p_pred_np[:, i], '--', label=f'pred p[{i}]', alpha=0.9)
axes[1].set_ylabel('p')
axes[1].set_xlabel('time')
axes[1].grid(True)
axes[1].legend(ncol=2, fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# --- 真値 vs モデル予測（同一初期値；直接推定）---

T_direct = 200
delta_t_vec = jnp.arange(T_direct) * dt

q0_batch = jnp.repeat(q0, repeats=T_direct, axis=0)
p0_batch = jnp.repeat(p0, repeats=T_direct, axis=0)

q_pred_direct, p_pred_direct, _, _ = model.apply(
    {'params': nn_params}, q0_batch, p0_batch, delta_t_vec, train=False
)

q_pred_direct_np = np.asarray(q_pred_direct)
p_pred_direct_np = np.asarray(p_pred_direct)

t_arr_direct = np.asarray(delta_t_vec)

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

for i in range(q_pred_direct_np.shape[1]):
    axes[0].plot(t_arr_direct, q_true_np[:T_direct, i], label=f'true q[{i}]', alpha=0.7)
    axes[0].plot(t_arr_direct, q_pred_direct_np[:, i], '--', label=f'direct pred q[{i}]', alpha=0.9)
axes[0].set_ylabel('q')
axes[0].grid(True)
axes[0].legend(ncol=2, fontsize=8)

for i in range(p_pred_direct_np.shape[1]):
    axes[1].plot(t_arr_direct, p_true_np[:T_direct, i], label=f'true p[{i}]', alpha=0.7)
    axes[1].plot(t_arr_direct, p_pred_direct_np[:, i], '--', label=f'direct pred p[{i}]', alpha=0.9)
axes[1].set_ylabel('p')
axes[1].set_xlabel('time')
axes[1].grid(True)
axes[1].legend(ncol=2, fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# evaluate test errors on test_dataset for each delta_t

x_test, _, y_test, delta_t_test = test_dataset
n_conf = x_test.shape[1] // 2

q_test = x_test[:, :n_conf]
p_test = x_test[:, n_conf:]
q_target = y_test[:, :n_conf]
p_target = y_test[:, n_conf:]

q_pred_test, p_pred_test, _, _ = model.apply(
    {'params': nn_params}, q_test, p_test, delta_t_test, train=False
)

per_sample_q_rmse = jnp.sqrt(jnp.mean((q_pred_test - q_target) ** 2, axis=1))
per_sample_p_rmse = jnp.sqrt(jnp.mean((p_pred_test - p_target) ** 2, axis=1))

delta_t_np = np.asarray(delta_t_test)
step_counts_per_sample = np.rint(delta_t_np / float(dt)).astype(int)
unique_steps = np.unique(step_counts_per_sample)

per_sample_q_rmse_np = np.asarray(per_sample_q_rmse)
per_sample_p_rmse_np = np.asarray(per_sample_p_rmse)
q_rmse_by_step = []
p_rmse_by_step = []
for step in unique_steps:
    mask = step_counts_per_sample == step
    q_rmse_by_step.append(per_sample_q_rmse_np[mask].mean())
    p_rmse_by_step.append(per_sample_p_rmse_np[mask].mean())

plt.figure(figsize=(7, 4))
plt.plot(unique_steps, q_rmse_by_step, 'o-', label='q RMSE')
plt.plot(unique_steps, p_rmse_by_step, 's-', label='p RMSE')
plt.xticks(unique_steps)
plt.xlabel('prediction horizon (steps)')
plt.ylabel('RMSE')
plt.grid(True, alpha=0.4)
plt.legend()
plt.title('Test error vs prediction horizon')
plt.show()
